## EDA and Data Cleaning

### Import Python Libraries and set properties for the notebook

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates

In [2]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
# Force pandas to display 2 decimal places (no scientific notation)
pd.options.display.float_format = '{:.2f}'.format

### Import data from the raw file

In [4]:
filepath = '../data/interim/neiss_interim_data.parquet'
data_interim = pd.read_parquet(filepath)

In [6]:
# Deep copy to preserve original data
data = data_interim.copy()
data1 = data_interim.copy()

In [7]:
data1.columns

Index(['data_year', 'CPSC_Case_Number', 'Treatment_Date', 'Age', 'Sex', 'Race',
       'Other_Race', 'Hispanic', 'Body_Part', 'Diagnosis', 'Other_Diagnosis',
       'Body_Part_2', 'Diagnosis_2', 'Other_Diagnosis_2', 'Disposition',
       'Location', 'Fire_Involvement', 'Product_1', 'Product_2', 'Product_3',
       'Alcohol', 'Drug', 'Narrative_1', 'Stratum', 'PSU', 'Weight',
       'True_Age'],
      dtype='str')

In [8]:
data1.info()

<class 'pandas.DataFrame'>
RangeIndex: 7315732 entries, 0 to 7315731
Data columns (total 27 columns):
 #   Column             Dtype         
---  ------             -----         
 0   data_year          int64         
 1   CPSC_Case_Number   int64         
 2   Treatment_Date     datetime64[us]
 3   Age                int64         
 4   Sex                int64         
 5   Race               int64         
 6   Other_Race         str           
 7   Hispanic           float64       
 8   Body_Part          int64         
 9   Diagnosis          int64         
 10  Other_Diagnosis    str           
 11  Body_Part_2        float64       
 12  Diagnosis_2        float64       
 13  Other_Diagnosis_2  str           
 14  Disposition        int64         
 15  Location           int64         
 16  Fire_Involvement   int64         
 17  Product_1          int64         
 18  Product_2          int64         
 19  Product_3          int64         
 20  Alcohol            float64       
 

### 1. Feature Engineering 

#### 1.1 Add Hospitalization Feature

To develop statistically sound predictive models to reliably identify factors associated with hospitalization of product-related injuries; we need to create a target variable for Hospitalized (Yes/No) for each injury reported based on the disposition code.

In [60]:
# B. Define Target Variable (Hospitalization)
# Disposition Codes:
# 1=Released, 2=Transferred, 4=Hospitalized, 5=Held for Obs, 6=Left, 8=Fatality
# We group 2, 4, 5, and 8 as "Severe/Hospitalized"
disposition_codes = [2, 4, 5, 8]
data1['Hospitalized'] = data1['Disposition'].apply(lambda x: 1 if x in disposition_codes else 0)

In [61]:
Hospitalized_df = data1.Hospitalized.value_counts(normalize=True).reset_index()
Hospitalized_df.columns = ['Hospitalized', 'Percentage']
print(Hospitalized_df)

   Hospitalized  Percentage
0             0        0.91
1             1        0.09


Data is unbalanced and we need to use any of the below techniques to balance the dataset.
1. Use the "class weight" parameter 
2. Use SMOTE (Oversampling) method 
3. Under Sampling - use all of the hospitalized data and take equal number of not_hospitalized data randomly. Could cause uncertain results.

#### 1.2 Add True Alcohol Flag

In [62]:
data1.Alcohol.value_counts()

Alcohol
3.00    5283928
0.00    1993098
1.00      38706
Name: count, dtype: int64

In [63]:
# Create a list of keywords (standard NEISS abbreviations)
alcohol_keywords = [
    # Medical & Formal
    'ALCOHOL', 'ETOH', 'INTOXICATED', 'INEBRIATED', 'DRUNK', 
    'SOBER', # Often appears as "Not Sober" or "Sobering up"
    'BAC ',  # Blood Alcohol Content (Need the space to avoid 'BACK')
    
    # Beverages
    'BEER', 'WINE', 'LIQUOR', 'VODKA', 'WHISKEY', 'WHISKY', 
    'TEQUILA', 'RUM', 'GIN', 'BRANDY', 'CHAMPAGNE', 'MOONSHINE',
    
    # Actions/Context
    'DRINKING', 'PARTY', 'BAR', 'PUB', 'CLUB', 'TAVERN',
    'HAPPY HOUR', 'KEG'
]

# Create a new column that searches the Narrative for these words
# Returns True if found, False if not
data1['Alcohol_byNarrative'] = data['Narrative_1'].str.contains('|'.join(alcohol_keywords), case=False, na=False)
data1['Alcohol_byNarrative'] = data1['Alcohol_byNarrative'].astype(float)

# Check how many you found
#data1['True_Alcohol_Flag'] = data1['Alcohol_bool'] | data1['Alcohol_Text_Flag']
print(data1['Alcohol_byNarrative'].value_counts())
print(data1['Alcohol'].value_counts())


Alcohol_byNarrative
0.00    6849504
1.00     466228
Name: count, dtype: int64
Alcohol
3.00    5283928
0.00    1993098
1.00      38706
Name: count, dtype: int64


In [64]:
target_cols = ['Alcohol', 'Alcohol_byNarrative']
data1['True_Alcohol'] = 0
mask_3 = data1[target_cols].eq(3.0).any(axis=1)
data1.loc[mask_3, 'True_Alcohol'] = 2
mask_1 = data1[target_cols].eq(1).any(axis=1)
data1.loc[mask_1, 'True_Alcohol'] = 1
print(data1['Alcohol_byNarrative'].value_counts())
print(data1['Alcohol'].value_counts())
print(data1['True_Alcohol'].value_counts())


Alcohol_byNarrative
0.00    6849504
1.00     466228
Name: count, dtype: int64
Alcohol
3.00    5283928
0.00    1993098
1.00      38706
Name: count, dtype: int64
True_Alcohol
2    4961221
0    1887004
1     467507
Name: count, dtype: int64


In [65]:
data1.drop(columns=['Alcohol_byNarrative'], inplace=True)

#### 1.3 Add True Drug Flag

In [66]:
# 1. Define keywords for common substances & slang
drug_keywords = [
    # General Terms
    'DRUG', 'OVERDOSE', 'MEDICATION', 'PILL', 'TABLET', 'CAPSULE', 
    'INGEST', 'SUBSTANCE', 'NARCOTIC', 'OPIOID', 'OPIGTE', # Common typo
    
    # Street / Illegal
    'COCAINE', 'HEROIN', 'METHAMPHETAMINE', 'MARIJUANA', 'CANNABIS', 
    'FENTANYL', 'ECSTASY', 'MDMA', ' LSD ', ' PCP ', 'HASHISH',
    
    # Prescription / OTC (Brand & Generic)
    'XANAX', 'VALIUM', 'PERCOCET', 'VICODIN', 'OXYCODONE', 'OXYCONTIN',
    'ADDERALL', 'RITALIN', 'TYLENOL', 'ASPIRIN', 'IBUPROFEN', 'ADVIL',
    'MOTRIN', 'BENADRYL', 'INSULIN', 'VITAMIN', 'MORPHINE', 'CODEINE'
]
# 2. Create the Search Pattern (joins them with OR logic: "DRUG|OVERDOSE|...")
pattern = '|'.join(drug_keywords)
# 3. Create a True/False column based on the text
data1['Drug_byNarrative'] = data1['Narrative_1'].str.contains(pattern, case=False, na=False)
data1['Drug_byNarrative'] = data1['Drug_byNarrative'].astype(float)

# Check how many you found
print(data1['Drug_byNarrative'].value_counts())
print(data1['Drug'].value_counts())


Drug_byNarrative
0.00    7165429
1.00     150303
Name: count, dtype: int64
Drug
3.00    5283928
0.00    1986756
1.00      45048
Name: count, dtype: int64


In [67]:
target_cols = ['Drug', 'Drug_byNarrative']
data1['True_Drug'] = 0
mask_3 = data1[target_cols].eq(3.0).any(axis=1)
data1.loc[mask_3, 'True_Drug'] = 2
mask_1 = data1[target_cols].eq(1).any(axis=1)
data1.loc[mask_1, 'True_Drug'] = 1
print(data1['Drug_byNarrative'].value_counts())
print(data1['Drug'].value_counts())
print(data1['True_Drug'].value_counts())

Drug_byNarrative
0.00    7165429
1.00     150303
Name: count, dtype: int64
Drug
3.00    5283928
0.00    1986756
1.00      45048
Name: count, dtype: int64
True_Drug
2    5190144
0    1952580
1     173008
Name: count, dtype: int64


In [68]:
data1.drop(columns=['Drug_byNarrative'], inplace=True)

### 3. Save the data to the interim folder for future use

In [ ]:
#data1.to_csv('../data/processed/neiss_processed_data.csv', index=False)
data1.to_parquet('../data/processed/neiss_processed_data.parquet')